# Education Data Warehouse — Data Profiling & ETL Prototype
**Mục tiêu:** đọc dữ liệu nguồn CSV, kiểm tra chất lượng, làm sạch và tạo các tệp Silver để tải lên BigQuery.  
**Lưu ý:** dữ liệu trong bộ này là dữ liệu mô phỏng, không đại diện cho một trung tâm thật. Chạy notebook trên Google Colab bằng cách upload thư mục `data/raw` hoặc mount Google Drive.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Colab: upload raw CSV files into /content/data/raw, or update this path.
RAW_DIR = Path('/content/data/raw')
SILVER_DIR = Path('/content/data/silver')
SILVER_DIR.mkdir(parents=True, exist_ok=True)

def read_csv(name):
    path = RAW_DIR / name
    if not path.exists():
        raise FileNotFoundError(f'Missing {path}. Upload the data/raw files first.')
    return pd.read_csv(path)


In [ ]:
students = read_csv('students.csv')
campaigns = read_csv('campaigns.csv')
leads = read_csv('leads.csv')
enrollments = read_csv('enrollments.csv')
courses = read_csv('courses.csv')
learning = read_csv('learning_records.csv')
guidance = read_csv('career_guidance.csv')

for name, df in {
    'students': students, 'campaigns': campaigns, 'leads': leads,
    'enrollments': enrollments, 'courses': courses,
    'learning': learning, 'guidance': guidance
}.items():
    print(f'{name:15s} rows={len(df):,} cols={len(df.columns)} missing_cells={int(df.isna().sum().sum()):,}')


FileNotFoundError: Missing /content/data/raw/students.csv. Upload the data/raw files first.

## 1. Quy tắc chất lượng dữ liệu
- Khóa định danh không được trống và cần duy nhất.
- Chuẩn hóa chữ hoa/thường và khoảng trắng.
- Không chấp nhận giá trị âm ở chi phí, giảm giá, số tiền và số lần liên hệ.
- Điểm đánh giá nằm trong khoảng 0–10; điểm hài lòng nằm trong khoảng 1–5.
- Loại bản ghi trùng khóa; kiểm tra khóa ngoại trước khi tạo bảng Fact.
- Giá trị không hợp lệ được đặt null hoặc loại khỏi bảng theo quy tắc đã ghi trong báo cáo; mọi thay đổi phải có log.


In [ ]:
# Cleaning helpers
def clean_students(df):
    d = df.copy()
    d['gender'] = d['gender'].fillna('Chưa xác định')
    d['district'] = d['district'].astype(str).str.strip().str.title()
    d['grade'] = d['grade'].astype(str).str.extract(r'(10|11|12)', expand=False)
    return d.drop_duplicates(subset=['student_id'])

def clean_campaigns(df):
    d = df.copy()
    d['marketing_spend'] = pd.to_numeric(d['marketing_spend'], errors='coerce')
    d.loc[d['marketing_spend'] < 0, 'marketing_spend'] = np.nan
    d['marketing_spend'] = d['marketing_spend'].fillna(d['marketing_spend'].median())
    return d.drop_duplicates(subset=['campaign_id'])

def clean_leads(df, valid_students, valid_campaigns):
    d = df.copy()
    d['lead_status'] = d['lead_status'].astype(str).str.strip().str.title()
    d['contact_attempts'] = pd.to_numeric(d['contact_attempts'], errors='coerce')
    d.loc[d['contact_attempts'] < 0, 'contact_attempts'] = np.nan
    d['contact_attempts'] = d['contact_attempts'].fillna(d['contact_attempts'].median()).astype(int)
    d = d[d.student_id.isin(valid_students) & d.campaign_id.isin(valid_campaigns)]
    return d.drop_duplicates(subset=['lead_id'])

def clean_enrollments(df, valid_students, valid_campaigns, valid_leads):
    d = df.copy()
    for c in ['list_price_vnd','discount_vnd','amount_paid_vnd']:
        d[c] = pd.to_numeric(d[c], errors='coerce')
        d.loc[d[c] < 0, c] = np.nan
    d['list_price_vnd'] = d['list_price_vnd'].fillna(d['list_price_vnd'].median())
    d['discount_vnd'] = d['discount_vnd'].fillna(0)
    d['amount_paid_vnd'] = d['amount_paid_vnd'].fillna(0)
    d['lead_id'] = d['lead_id'].where(d['lead_id'].isin(valid_leads), None)
    d = d[d.student_id.isin(valid_students) & d.campaign_id.isin(valid_campaigns)]
    return d.drop_duplicates(subset=['enrollment_id'])

def clean_learning(df, valid_enrollments, valid_students, valid_courses):
    d = df.copy()
    d['attendance_status'] = d['attendance_status'].astype(str).str.strip().str.title()
    d['assessment_score'] = pd.to_numeric(d['assessment_score'], errors='coerce')
    d.loc[~d.assessment_score.between(0,10), 'assessment_score'] = np.nan
    d = d[d.enrollment_id.isin(valid_enrollments) & d.student_id.isin(valid_students) & d.course_id.isin(valid_courses)]
    return d.drop_duplicates(subset=['learning_record_id'])

def clean_guidance(df, valid_students):
    d = df.copy()
    d['satisfaction_score'] = pd.to_numeric(d['satisfaction_score'], errors='coerce')
    d.loc[~d.satisfaction_score.between(1,5), 'satisfaction_score'] = np.nan
    return d[d.student_id.isin(valid_students)].drop_duplicates(subset=['guidance_id'])


In [ ]:
silver_students = clean_students(students)
silver_campaigns = clean_campaigns(campaigns)
silver_leads = clean_leads(leads, set(silver_students.student_id), set(silver_campaigns.campaign_id))
silver_enrollments = clean_enrollments(
    enrollments, set(silver_students.student_id), set(silver_campaigns.campaign_id), set(silver_leads.lead_id)
)
silver_courses = courses.drop_duplicates('course_id')
silver_learning = clean_learning(
    learning, set(silver_enrollments.enrollment_id), set(silver_students.student_id), set(silver_courses.course_id)
)
silver_guidance = clean_guidance(guidance, set(silver_students.student_id))

cleaned = {
    'students_clean.csv': silver_students,
    'campaigns_clean.csv': silver_campaigns,
    'leads_clean.csv': silver_leads,
    'enrollments_clean.csv': silver_enrollments,
    'courses_clean.csv': silver_courses,
    'learning_records_clean.csv': silver_learning,
    'career_guidance_clean.csv': silver_guidance
}
for name, df in cleaned.items():
    df.to_csv(SILVER_DIR / name, index=False)
    print(name, len(df), 'rows')


In [ ]:
# Quality checks and reproducible local KPIs (based on synthetic data)
assert silver_students.student_id.is_unique
assert silver_leads.lead_id.is_unique
assert silver_enrollments.enrollment_id.is_unique
assert silver_learning.learning_record_id.is_unique
assert silver_guidance.guidance_id.is_unique
assert silver_learning.assessment_score.dropna().between(0,10).all()
assert silver_guidance.satisfaction_score.dropna().between(1,5).all()

lead_count = len(silver_leads)
converted_count = int((silver_leads.lead_status == 'Converted').sum())
conversion_rate = converted_count / lead_count if lead_count else np.nan
recognized_revenue = silver_enrollments.loc[
    ~silver_enrollments.enrollment_status.isin(['Cancelled','Refunded']), 'amount_paid_vnd'
].sum()
attendance_rate = (silver_learning.attendance_status == 'Present').mean()

print({
    'clean_leads': lead_count,
    'converted_leads': converted_count,
    'lead_conversion_rate': round(float(conversion_rate), 4),
    'recognized_revenue_vnd': int(recognized_revenue),
    'attendance_rate': round(float(attendance_rate), 4)
})


In [ ]:
# Simple charts for the report / presentation
import matplotlib.pyplot as plt

monthly = silver_enrollments.copy()
monthly['enrollment_date'] = pd.to_datetime(monthly['enrollment_date'], errors='coerce')
monthly['month'] = monthly['enrollment_date'].dt.to_period('M').astype(str)
monthly.groupby('month').size().plot(kind='bar', figsize=(12,4), title='Enrollments by month (synthetic data)')
plt.xlabel('Month'); plt.ylabel('Enrollment records'); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

silver_leads['lead_status'].value_counts().plot(kind='bar', figsize=(8,4), title='Lead status distribution (synthetic data)')
plt.xlabel('Lead status'); plt.ylabel('Number of leads'); plt.tight_layout(); plt.show()


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import glob
import os

SILVER_DIR = "/content/drive/MyDrive/data/silver"

csv_files = sorted(glob.glob(os.path.join(SILVER_DIR, "*.csv")))

print("Số file CSV:", len(csv_files))
for f in csv_files:
    print(os.path.basename(f))

Mounted at /content/drive
Số file CSV: 0


In [2]:
from google.colab import auth
from google.cloud import bigquery
import os
import re

auth.authenticate_user()

PROJECT_ID = "data-warehouse-512306"
DATASET_ID = "education_dw"

client = bigquery.Client(project=PROJECT_ID)

for file_path in csv_files:
    filename = os.path.splitext(os.path.basename(file_path))[0]

    # Ví dụ campaigns_clean -> silver_campaigns
    base_name = re.sub(r"_clean$", "", filename)
    table_name = "silver_" + base_name

    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
    )

    with open(file_path, "rb") as f:
        job = client.load_table_from_file(
            f,
            table_id,
            job_config=config
        )

    job.result()

    table = client.get_table(table_id)
    print(f"OK: {table_id} | {table.num_rows} dòng")

In [4]:
import os

root = "/content/drive/MyDrive"
project_folder = None

# Tìm thư mục dự án
for current_dir, dirs, files in os.walk(root):
    for d in dirs:
        if "PROJECT DATA WAREHOUSE" in d.upper():
            project_folder = os.path.join(current_dir, d)
            break
    if project_folder:
        break

print("Thư mục dự án:", project_folder)

if project_folder:
    print("\nCác file CSV bên trong dự án:")
    for current_dir, dirs, files in os.walk(project_folder):
        for name in files:
            if name.lower().endswith(".csv"):
                print(os.path.join(current_dir, name))
else:
    print("Chưa tìm thấy thư mục dự án.")

Thư mục dự án: /content/drive/MyDrive/PROJECT DATA WAREHOUSE

Các file CSV bên trong dự án:
/content/drive/MyDrive/PROJECT DATA WAREHOUSE/Education_DW_Project_Bundle/Education_DW_Project/data/silver/learning_records_clean.csv
/content/drive/MyDrive/PROJECT DATA WAREHOUSE/Education_DW_Project_Bundle/Education_DW_Project/data/silver/enrollments_clean.csv
/content/drive/MyDrive/PROJECT DATA WAREHOUSE/Education_DW_Project_Bundle/Education_DW_Project/data/silver/courses_clean.csv
/content/drive/MyDrive/PROJECT DATA WAREHOUSE/Education_DW_Project_Bundle/Education_DW_Project/data/silver/career_guidance_clean.csv
/content/drive/MyDrive/PROJECT DATA WAREHOUSE/Education_DW_Project_Bundle/Education_DW_Project/data/silver/leads_clean.csv
/content/drive/MyDrive/PROJECT DATA WAREHOUSE/Education_DW_Project_Bundle/Education_DW_Project/data/silver/campaigns_clean.csv
/content/drive/MyDrive/PROJECT DATA WAREHOUSE/Education_DW_Project_Bundle/Education_DW_Project/data/silver/students_clean.csv
/content/dr

In [5]:
from google.colab import auth
from google.cloud import bigquery
import os
import glob
import re

# 1. Kết nối Google Cloud
auth.authenticate_user()

PROJECT_ID = "data-warehouse-512306"
DATASET_ID = "education_dw"

client = bigquery.Client(project=PROJECT_ID)

# 2. Đường dẫn Silver chính xác
SILVER_DIR = (
    "/content/drive/MyDrive/PROJECT DATA WAREHOUSE/"
    "Education_DW_Project_Bundle/Education_DW_Project/"
    "data/silver"
)

csv_files = sorted(glob.glob(os.path.join(SILVER_DIR, "*.csv")))

if len(csv_files) != 7:
    raise ValueError(
        f"Mong đợi 7 file Silver nhưng tìm thấy {len(csv_files)}. "
        "Kiểm tra lại đường dẫn trước khi tiếp tục."
    )

# 3. Kiểm tra dataset
dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
client.get_dataset(dataset_id)

# 4. Nạp các file Silver
results = []

for file_path in csv_files:
    filename = os.path.splitext(os.path.basename(file_path))[0]
    base_name = re.sub(r"_clean$", "", filename)
    table_name = f"silver_{base_name}"
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"

    # Không ghi đè bảng đã tồn tại
    try:
        existing_table = client.get_table(table_id)
        print(
            f"ĐÃ TỒN TẠI: {table_name} | "
            f"{existing_table.num_rows} dòng — bỏ qua"
        )
        results.append(
            (table_name, existing_table.num_rows, "Đã tồn tại")
        )
        continue
    except Exception as e:
        from google.api_core.exceptions import NotFound
        if not isinstance(e, NotFound):
            raise

    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,
        autodetect=True,
        create_disposition=bigquery.CreateDisposition.CREATE_IF_NEEDED,
        write_disposition=bigquery.WriteDisposition.WRITE_EMPTY,
    )

    print(f"Đang nạp {filename}...")

    with open(file_path, "rb") as f:
        job = client.load_table_from_file(
            f,
            table_id,
            job_config=job_config
        )

    job.result()

    table = client.get_table(table_id)

    print(f"THÀNH CÔNG: {table_name} | {table.num_rows} dòng")
    results.append((table_name, table.num_rows, "Đã nạp"))

print("\n===== TỔNG KẾT =====")
for name, rows, status in results:
    print(f"{name}: {rows} dòng | {status}")

Đang nạp campaigns_clean...
THÀNH CÔNG: silver_campaigns | 24 dòng
Đang nạp career_guidance_clean...
THÀNH CÔNG: silver_career_guidance | 259 dòng
Đang nạp courses_clean...
THÀNH CÔNG: silver_courses | 6 dòng
Đang nạp enrollments_clean...
THÀNH CÔNG: silver_enrollments | 390 dòng
Đang nạp leads_clean...
THÀNH CÔNG: silver_leads | 899 dòng
Đang nạp learning_records_clean...
THÀNH CÔNG: silver_learning_records | 2502 dòng
Đang nạp students_clean...
THÀNH CÔNG: silver_students | 500 dòng

===== TỔNG KẾT =====
silver_campaigns: 24 dòng | Đã nạp
silver_career_guidance: 259 dòng | Đã nạp
silver_courses: 6 dòng | Đã nạp
silver_enrollments: 390 dòng | Đã nạp
silver_leads: 899 dòng | Đã nạp
silver_learning_records: 2502 dòng | Đã nạp
silver_students: 500 dòng | Đã nạp


## 2. Nạp lên BigQuery (thực hiện thủ công sau khi tạo project)
1. Tạo Google Cloud Project và bật BigQuery API.
2. Mở BigQuery Studio, tạo datasets `education_dw_bronze`, `education_dw_silver`, `education_dw_gold`.
3. Upload các CSV trong `data/raw` vào Bronze; cấu hình schema phù hợp và đánh dấu dòng tiêu đề.
4. Upload các CSV trong `data/silver` vào Silver nếu muốn so sánh với pipeline dbt; hoặc dùng dbt staging để biến đổi trực tiếp từ Bronze.
5. Thay `YOUR_GCP_PROJECT_ID` trong các tệp SQL/YAML bằng Project ID thật.
6. Cài dbt BigQuery và xác thực bằng OAuth/Application Default Credentials. Không tải khóa JSON lên notebook hoặc Git.
7. Chạy `dbt debug`, `dbt run`, `dbt test`; lưu ảnh chụp kết quả thật vào báo cáo.
8. Chưa chạy các bước trên thì không ghi rằng dữ liệu đã được triển khai thành công trên BigQuery.
